# 1. Introduction to Image Captioning with VizWiz


## 1.1 Motivation and Model Intuition

In the field of **Computer Vision** and **Natural Language Processing (NLP)**, image captioning represents a bridge between visual understanding and linguistic expression. The primary objective of this project is to develop a deep learning model capable of generating descriptive text for images.

This technology has profound real-world applications in **Assistive Technology**. By automating the description of visual environments, we can build accessibility tools that assist individuals who are blind or have low vision. We are utilizing the **VizWiz dataset**, which is specifically designed to reflect the challenges faced by this community.

**The Encoder-Decoder Architecture**

We implement a two-part **Encoder-Decoder** architecture using **PyTorch**:
1.  **The Encoder (The "Eyes")**: A pre-trained **Convolutional Neural Network (CNN)** (e.g., 'ResNet', 'VGG', or 'Inception'). Its job is to compress raw pixel data into a dense, high-dimensional feature vector representing the image's core visual semantic information.
2.  **The Decoder (The "Voice")**: A sequence-based model such as a **Recurrent Neural Network (RNN)**, **Long Short-Term Memory (LSTM)**, or **Gated Recurrent Unit (GRU)**. It takes the visual features and predicts a sequence of tokens, learning the **conditional probability** of a word given the image context and all previously generated words.

## 1.2 Challenge Definition and Evaluation Metrics

The core task is **Image Captioning**: mapping a high-dimensional image input space to a discrete, sequential text output space.

To evaluate the quality of our generated captions, we use the **BLEU (Bilingual Evaluation Understudy)** metric. BLEU measures the **n-gram overlap** between the machine-generated caption and the five human-provided reference captions. We will track four variations:
*   **BLEU-1**: Measures individual word accuracy (unigrams).
*   **BLEU-2, 3, 4**: Measure the fluency and coherence of phrases (bigrams, trigrams, and 4-grams).

While BLEU provides a quantitative baseline, we will also perform **Qualitative Analysis** through visual inspection of the model's predictions.

## 1.3 Project Instructions and Workflow

To maintain **MLOps** best practices, our project follows a strictly phased workflow. This promotes collaboration while ensuring individual accountability for model architecture design.

**Phase 1: Collaborative Data Preparation**

The group executes a shared notebook to handle:
*   Downloading the **VizWiz** dataset.
*   Cleaning and **tokenizing** captions.
*   Building a mapping of words to indices (the **Vocabulary**).
*   Constructing PyTorch `DataLoaders`.

**Phase 2: Independent Modeling (Model 1)**

Each member independently designs, trains, and evaluates their first unique architecture. You are encouraged to experiment with different **CNN backbones** and **RNN variants**.

**Phase 3: Review and Refine (Model 2)**

The group analyzes the results of all "Model 1" versions. Each member then builds a **Refined Architecture**, incorporating the collective insights gained from the initial training runs.


## 1.4 Dataset Overview: VizWiz-Captions

We are utilizing the **validation** split of the **VizWiz-Captions** dataset. Unlike academic datasets such as MS COCO, VizWiz images are captured by people with visual impairments.

### Dataset Characteristics:
*   **Volume**: 7,750 images.
*   **Annotations**: 38,750 total captions (5 human-generated captions per image).
*   **Real-world Complexity**: These images often feature **poor lighting**, **motion blur**, and **unconventional framing**. These "in-the-wild" artifacts make the captioning task significantly more difficult but highly authentic for accessibility applications.

# 2. Initialization


## 2.1 Environment Setup and Logging

In [ ]:
import os
import platform
import sys

def log_environment_info() -> None:
    """Logs the system and Python environment details."""
    print(f'Python : {sys.version}')
    print(f'OS     : {platform.system()} {platform.release()}')
    print(f'CWD    : {os.getcwd()}')

log_environment_info()

## 2.2 Libraries

Modern Deep Learning relies on a robust ecosystem of libraries. We utilize **PyTorch** for computational graphs and tensor operations, **NumPy** for numerical processing, and **Pandas** for structured data manipulation. We also implement a **Warning Filter** to suppress non-critical engine warnings, ensuring that our final notebook remains professional and readable for peer review or production handover.

In [ ]:
import warnings
import gc
import random
from typing import List, Union
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Suppress warnings for cleaner output presentation
warnings.filterwarnings('ignore')

def check_library_versions() -> None:
    """Prints the versions of the core data science libraries."""
    print(f'PyTorch : {torch.__version__}')
    print(f'NumPy   : {np.__version__}')
    print(f'Pandas  : {pd.__version__}')

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

check_library_versions()

## 2.3 Centralized Configuration (Settings)

A best practice in high-level Data Science is the **Singleton Configuration Pattern**. Instead of hardcoding hyperparameters like 'LEARNING_RATE' or 'BATCH_SIZE' inside functions, we centralize them in a `Settings` class. This facilitates **Hyperparameter Optimization (HPO)** and makes switching between 'train' and 'inference' modes seamless. We use `pathlib` for **Object-Oriented Filesystem Paths**, which is more robust than standard string manipulation.

In [ ]:
class Settings:
    """Centralized configuration class for the Image Captioning project.

    Contains all hyperparameters, environment settings, and dynamic path
    routing for both Kaggle and local VS Code environments.
    """

    # Operational Toggles
    MODE: str = "train"
    WANDB: bool = False
    DEBUG: bool = False

    # Environment Detection
    IS_KAGGLE: bool = Path("/kaggle/input").exists()

    # Dynamic Path Management based on environment
    if IS_KAGGLE:
        # Kaggle read-only input paths based on uploaded structure
        DATA_DIR = Path("/kaggle/input/datasets/tuannm3823/vizwiz")
        ANNOTATIONS_DIR = DATA_DIR / "annotations" / "annotations"
        TRAIN_IMG_DIR = DATA_DIR / "train" / "train"
        VAL_IMG_DIR = DATA_DIR / "val" / "val"
        TEST_IMG_DIR = DATA_DIR / "test" / "test"

        # Kaggle writable working directory
        WORK_DIR = Path("/kaggle/working")
    else:
        # Local VS Code paths based on uploaded structure
        DATA_DIR = Path("./data")
        ANNOTATIONS_DIR = DATA_DIR / "annotations"
        TRAIN_IMG_DIR = DATA_DIR / "images" / "train" / "train"
        VAL_IMG_DIR = DATA_DIR / "images" / "val" / "val"
        TEST_IMG_DIR = DATA_DIR / "images" / "test" / "test"

        # Local writable working directory
        WORK_DIR = DATA_DIR / "working"

    BASE: Path = DATA_DIR
    CACHE_DIR: Path = WORK_DIR / "cache"
    WORK_CACHE_DIR: Path = CACHE_DIR / "work"

    # Model Hyperparameters
    SEED: int = 42
    LEARNING_RATE: float = 1e-4
    BATCH_SIZE: int = 32
    EPOCHS: int = 10
    MAX_SEQ_LEN: int = 50

# Instantiate configuration object
CFG = Settings()

# Post-instantiation directory setup
CFG.CACHE_EXISTS = CFG.CACHE_DIR.exists()
CFG.WORK_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'MODE         : {CFG.MODE}')
print(f'BASE         : {CFG.BASE}')
print(f'CACHE_EXISTS : {CFG.CACHE_EXISTS}')
print(f'CACHE_DIR    : {CFG.CACHE_DIR}')

## 2.4 Memory Optimization and Management

Deep Learning models are notorious for **RAM/VRAM** exhaustion. To mitigate **Out-of-Memory (OOM)** errors, we implement two strategies:
1.  **Garbage Collection**: Manually triggering `gc.collect()` and `torch.cuda.empty_cache()` to free unused pointers.
2.  **Downcasting**: The `reduce_mem_usage` function iterates through a **Pandas DataFrame** and converts data types to their smallest possible representation (e.g., `float64` to `float32` or `int64` to `int8`).


In [ ]:
def clear_memory() -> None:
    """Forces garbage collection and clears PyTorch CUDA cache to prevent OOM."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Memory cleared.")

def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    """
    Iterates through DataFrame columns and downcasts data types to save memory.

    Args:
        df (pd.DataFrame): The input DataFrame to compress.

    Returns:
        pd.DataFrame: The memory-optimized DataFrame.
    """
    start_mem = df.memory_usage().sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()

            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage().sum() / 1024**2
    if CFG.DEBUG:
        print(f'Mem. usage decreased to {end_mem:.2f} Mb ({(100 * (start_mem - end_mem) / start_mem):.1f}% reduction)')

    return df

## 2.5 Seeding

Deep learning is inherently **stochastic**. Weights are initialized randomly, and data is shuffled. To ensure that our performance metrics are comparable across different runs and by different team members, we must enforce **Global Determinism**. We do this by locking the seeds for the Python `random` module, `numpy`, and `torch`. Crucially, we also disable the **cuDNN benchmark** and set it to deterministic mode to ensure identical results on GPU backends.

In [ ]:
def seed_everything(seed: int = 42) -> None:
    """
    Sets the seed for reproducibility across Python, NumPy, and PyTorch.
    Ensures deterministic behavior for GPU operations.

    Args:
        seed (int): The numerical seed value.
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # cuDNN determinism: ensures results are reproducible on the same GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    print(f"Global seed locked to: {seed}")

# Initialize global seed for Phase 1
seed_everything(CFG.SEED)

# 3. Data Preparing


## 3.1 Data Loading

In modern **MLOps pipelines**, manual data handling is a significant bottleneck and a primary source of environment drift. To ensure **reproducibility** and scalability, we must automate the acquisition of our raw assets. The **VizWiz dataset** consists of high-resolution images and complex JSON annotations, which are often too large to fit entirely into system memory (RAM) during the download phase.

By employing **chunked streaming**, we download the data in small, manageable pieces (typically 8KB to 1MB). This approach prevents **Out-of-Memory (OOM)** errors and allows us to handle multi-gigabyte ZIP files efficiently. Furthermore, organizing these files into a structured `raw_data` directory ensures that the downstream preprocessing steps have a consistent and predictable data source.

In [ ]:
import os
import zipfile
from pathlib import Path
from typing import Dict

import requests

def download_data(url: str, dest_dir: Path) -> None:
    """Downloads a ZIP file from a URL.

    Args:
        url (str): The direct URL to the zip file.
        dest_dir (Path): The directory to save the contents.
    """
    dest_dir.mkdir(parents=True, exist_ok=True)
    filename = url.split("/")[-1]
    zip_path = dest_dir / filename

    if not zip_path.exists():
        print(f"Downloading {filename}...")
        response = requests.get(url, stream=True)
        response.raise_for_status()
        with open(zip_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    file.write(chunk)
        print(f"Download of {filename} complete.")
    else:
        print(f"{filename} already exists. Skipping download.")


def prepare_data() -> None:
    """Prepares data by downloading only if missing locally.

    Bypasses download on Kaggle since data is mounted as a dataset.
    Assumes data is already extracted as per user instruction.
    """
    if CFG.IS_KAGGLE:
        print("Kaggle environment detected. Using pre-uploaded datasets.")
        return

    url_dict: Dict[str, str] = {
        "train": "https://vizwiz.cs.colorado.edu/VizWiz_final/images/train.zip",
        "val": "https://vizwiz.cs.colorado.edu/VizWiz_final/images/val.zip",
        "test": "https://vizwiz.cs.colorado.edu/VizWiz_final/images/test.zip",
        "annotations": "https://vizwiz.cs.colorado.edu/VizWiz_final/caption/annotations.zip",
    }

    for split_name, download_url in url_dict.items():
        if split_name == "annotations":
            target_dir = CFG.ANNOTATIONS_DIR
        else:
            # Directing image zips to the images folder to match local struct
            target_dir = CFG.DATA_DIR / "images" / split_name

        # Check if the deepest nested directory exists
        expected_dir = CFG.TRAIN_IMG_DIR if split_name == "train" else target_dir

        if not expected_dir.exists():
            print(f"\n--- Processing {split_name.upper()} Split ---")
            download_data(download_url, target_dir)
            print(f"Data for {split_name} downloaded. Manual extraction required.")
        else:
            print(f"Data for {split_name} already exists. Skipping.")


if __name__ == "__main__":
    print(f"MODE         : {CFG.MODE}")
    print(f"IS_KAGGLE    : {CFG.IS_KAGGLE}")
    print(f"ANNOTATIONS  : {CFG.ANNOTATIONS_DIR}")
    print(f"TRAIN_IMG    : {CFG.TRAIN_IMG_DIR}")
    print(f"WORK_DIR     : {CFG.WORK_DIR}")

    prepare_data()

## 3.2 Parsing and Cleaning Annotations

In Vision-Language (VL) tasks, the bridge between visual features and linguistic tokens is the **Metadata Annotation**. The VizWiz dataset follows a relational JSON structure (similar to COCO) where image metadata and textual captions are stored in separate lists linked by unique identifiers.

When preparing data for deep learning, we must address two critical factors:
1.  **Textual Integrity**: Legacy characters such as `\r` (carriage returns) can interfere with modern **Byte-Pair Encoding (BPE)** or **WordPiece** tokenizers used by models like CLIP or BLIP, leading to unexpected "unknown" tokens or misalignment.
2.  **Lookup Efficiency**: Converting these JSON structures into a **Pandas DataFrame** allows for vectorized operations and $O(1)$ access times during the `__getitem__` calls in our PyTorch `Dataset` class.

We will also implement a **Memory Optimization** pass. By downcasting numeric types (e.g., `int64` to `int32`), we reduce the RAM footprint of our 'val_df', which is essential when working with large-scale datasets in memory-constrained environments like Google Colab.

In [ ]:
# Solution
import json
from pathlib import Path
import pandas as pd

def load_and_merge_annotations(data_dir: Path, split: str = 'train') -> pd.DataFrame:
    """Loads VizWiz JSON annotations, cleans text, and merges images with captions.

    Args:
        data_dir (Path): The base data directory (e.g., './data').
        split (str): The dataset split to load ('train' or 'val').

    Returns:
        pd.DataFrame: A merged DataFrame containing file names and captions.
    """
    # Navigate to the exact folder structure shown in your environment
    json_path = CFG.ANNOTATIONS_DIR / f"{split}.json"

    print(f"Loading {split} annotations from: {json_path}")
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Convert the JSON lists into Pandas DataFrames
    images_df = pd.DataFrame(data['images'])
    annotations_df = pd.DataFrame(data['annotations'])

    # Clean captions: Remove legacy '\r' characters to prevent tokenization misalignment
    annotations_df['caption'] = annotations_df['caption'].str.replace('\r', '', regex=False)

    # Merge annotations with image file names using their unique IDs
    merged_df = pd.merge(
        annotations_df,
        images_df[['id', 'file_name']],
        left_on='image_id',
        right_on='id',
        how='inner'
    )

    # Clean up redundant columns
    merged_df = merged_df.drop(columns=['id_y']).rename(columns={'id_x': 'annotation_id'})

    # Optimize memory using our utility function from Section 2
    merged_df = reduce_mem_usage(merged_df)

    return merged_df

# Define our base data path
base_data_path = Path("./data")

# Load the training data (matching the expanded folder in your screenshot)
train_df = load_and_merge_annotations(base_data_path, split='train')
val_df = load_and_merge_annotations(base_data_path, split='val')

# Apply debugging toggle if activated in our CFG class
if CFG.DEBUG:
    print("DEBUG MODE: Sampling 500 rows for rapid testing.")
    train_df = train_df.sample(500, random_state=CFG.SEED).reset_index(drop=True)

print(f"\nTotal processed {train_df.shape[0]} training annotations.")
display(train_df[['file_name', 'caption', 'text_detected', 'is_rejected']].head())

In [ ]:
display(train_df.head())

In [ ]:
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def show_sample_with_plotly(df: pd.DataFrame, image_dir: Path, num_samples: int = 1):
    """Displays images and captions using an interactive Plotly subplot."""

    # Group by image to get all 5 captions per file
    grouped = df.groupby('file_name')
    unique_files = list(grouped.groups.keys())

    # Pick random samples
    samples = random.sample(unique_files, k=num_samples)

    for file_name in samples:
        group = grouped.get_group(file_name)
        img_path = image_dir / file_name

        # 1. Prepare Table Data and Colors
        captions = []
        colors = []

        for _, row in group.iterrows():
            if row['is_rejected']:
                captions.append(f"[REJECTED] {row['caption']}")
                colors.append('red')
            elif row['is_precanned']:
                captions.append(f"[PRECANNED] {row['caption']}")
                colors.append('royalblue')
            else:
                captions.append(row['caption'])
                colors.append('black')

        # 2. Create Subplots (Image Left, Table Right)
        fig = make_subplots(
            rows=1, cols=2,
            column_widths=[0.4, 0.6],
            specs=[[{"type": "image"}, {"type": "table"}]],
            horizontal_spacing=0.05
        )

        # 3. Add Image Trace
        if img_path.exists():
            # Plotly go.Image requires a numpy array
            img_array = np.array(Image.open(img_path))
            fig.add_trace(go.Image(z=img_array), row=1, col=1)

        # 4. Add Table Trace
        fig.add_trace(
            go.Table(
                header=dict(
                    values=[f"<b>Captions for {file_name}</b>"],
                    align="left",
                    font=dict(color="white", size=14),
                    fill_color="#2C3E50" # Dark slate blue header
                ),
                cells=dict(
                    # Values wrapped in a list because it's a single column table
                    values=[captions],
                    align="left",
                    # Pass the dynamic color list we built above
                    font=dict(color=[colors], size=13),
                    height=40
                )
            ),
            row=1, col=2
        )

        # 5. Clean up layout (hide axes for the image)
        fig.update_layout(
            height=400,
            margin=dict(l=10, r=10, t=30, b=10),
            plot_bgcolor="white"
        )
        fig.update_xaxes(visible=False, row=1, col=1)
        fig.update_yaxes(visible=False, row=1, col=1)

        fig.show()

# Execute for the train directory
train_image_dir = CFG.TRAIN_IMG_DIR
show_sample_with_plotly(train_df, train_image_dir, num_samples=3)

## 3.3 Building the Vocabulary

In Computer Vision and Natural Language Processing (CV-NLP) tasks like Image Captioning, the model cannot directly interpret raw strings. We must map every unique word to a specific integer representation, a process known as **Numericalization**.

To manage this, we implement a custom `Vocabulary` class. This utility serves several critical MLOps and Data Science purposes:
1.  **Special Token Handling**: We reserve specific IDs for **Special Tokens** such as `<PAD>` (padding sequences to equal length), `<START>`/`<END>` (marking sequence boundaries), and `<UNK>` (Unknown tokens for words not seen during training).
2.  **Frequency Thresholding**: By using a `freq_threshold`, we filter out rare words or typos. This reduces the **Dimensionality** of our embedding layer and prevents the model from overfitting on noise.
3.  **Bidirectional Mapping**: We maintain two dictionaries, 'stoi' (**String-to-Index**) for encoding and 'itos' (**Index-to-String**) for decoding model predictions back into human-readable text.

In [ ]:
from collections import Counter
from typing import List, Dict

class Vocabulary:
    """
    Handles the mapping between words and their corresponding integer IDs.

    Attributes:
        freq_threshold (int): Minimum occurrences required for a word to be indexed.
        itos (Dict[int, str]): Index-to-string mapping.
        stoi (Dict[str, int]): String-to-index mapping.
    """

    def __init__(self, freq_threshold: int = 2) -> None:
        self.freq_threshold: int = freq_threshold

        # Initialize dictionaries with reserved special tokens
        # 0: Padding, 1: Start of sentence, 2: End of sentence, 3: Unknown word
        self.itos: Dict[int, str] = {0: "<PAD>", 1: "<START>", 2: "<END>", 3: "<UNK>"}
        self.stoi: Dict[str, int] = {"<PAD>": 0, "<START>": 1, "<END>": 2, "<UNK>": 3}
        self.idx: int = 4

    def __len__(self) -> int:
        """Returns the total number of unique words in the vocabulary."""
        return len(self.itos)

    def tokenize(self, text: str) -> List[str]:
        """
        Performs basic text cleaning and tokenization.

        Args:
            text (str): Raw caption string.

        Returns:
            List[str]: List of lowercase tokens with isolated punctuation.
        """
        text = text.lower()
        # Ensure punctuation is treated as separate tokens to improve context learning
        for punc in [".", ",", "!", "?", '"', "'", "(", ")"]:
            text = text.replace(punc, f" {punc} ")
        return text.split()

    def build_vocabulary(self, sentence_list: List[str]) -> None:
        """
        Iterates through the dataset to build the stoi and itos mappings.

        Args:
            sentence_list (List[str]): A list of all training captions.
        """
        frequencies: Counter = Counter()

        for sentence in sentence_list:
            for word in self.tokenize(sentence):
                frequencies[word] += 1

        # Only add words to vocabulary if they meet the frequency requirement
        for word, count in frequencies.items():
            if count >= self.freq_threshold:
                self.stoi[word] = self.idx
                self.itos[self.idx] = word
                self.idx += 1

    def numericalize(self, text: str) -> List[int]:
        """
        Converts a raw string into a list of integer indices.

        Args:
            text (str): Caption string.

        Returns:
            List[int]: Numerical representation of the text.
        """
        tokenized_text = self.tokenize(text)

        return [
            self.stoi[token] if token in self.stoi else self.stoi["<UNK>"]
            for token in tokenized_text
        ]

# 1. Extract all captions from the training DataFrame to build the corpus
# Assuming 'train_df' is already loaded and contains a 'caption' column
all_train_captions: List[str] = train_df['caption'].tolist()

# 2. Initialize and build the vocabulary
# A threshold of 2 helps ignore rare outliers and potential data entry errors
vocab = Vocabulary(freq_threshold=2)
vocab.build_vocabulary(all_train_captions)

print(f"Total reference captions: {len(all_train_captions)}")
print(f"Total vocabulary size: {len(vocab)} words")

# 3. Test the numericalization pipeline on a sample caption
sample_idx: int = 0
sample_text: str = all_train_captions[sample_idx]

print(f"\n[Validation] Sample Text: {sample_text}")
print(f"[Validation] Tokenized: {vocab.tokenize(sample_text)}")
print(f"[Validation] Numericalized: {vocab.numericalize(sample_text)}")

## 3.4 Data Quality Audit

Before finalizing our data pipeline, we must verify that our merging logic produced a perfectly clean 'pd.DataFrame'. A robust audit ensures **Data Integrity** by checking for **null values**, **duplicate entries**, and **structural consistency**. In the context of Image Captioning, it is vital to confirm the many-to-one relationship between captions and images (standard datasets like COCO typically maintain a 5:1 ratio).


In [ ]:
def audit_dataset(df: pd.DataFrame, split_name: str = "Train") -> None:
    """
    Performs rigorous QA checks on the dataset to ensure MLOps compliance.

    This function validates the integrity of the dataframe by checking for
    missing values, duplicate IDs, empty strings, and the required
    image-to-caption distribution ratio.

    Args:
        df (pd.DataFrame): The DataFrame containing image-caption pairs.
        split_name (str): The label of the dataset split (e.g., 'Train', 'Val').
    """
    print(f"--- Data Quality Audit: {split_name} Set ---\n")

    # 1. Check for missing (NaN) values across the entire DataFrame
    # Missing data can cause crashes during tensor conversion or loss calculation.
    missing_vals = df.isnull().sum().sum()
    print(f"[Check 1] Missing Values (NaN): {missing_vals}")

    # 2. Check for duplicate unique identifiers
    # Duplicate 'annotation_id' values suggest errors in the data join/merge logic.
    duplicate_ids = df['annotation_id'].duplicated().sum()
    print(f"[Check 2] Duplicate Annotation IDs: {duplicate_ids}")

    # 3. Check for empty strings or purely whitespace captions
    # Empty strings result in NaN losses during Natural Language Processing (NLP) tasks.
    empty_captions = (df['caption'].astype(str).str.strip() == '').sum()
    print(f"[Check 3] Empty/Whitespace Captions: {empty_captions}")

    # 4. Verify the ratio of captions per image
    # For balanced training, every image must ideally have the same number of captions.
    captions_per_image = df.groupby('file_name').size()
    print("\n[Check 4] Caption Counts per Image Distribution:")
    print(captions_per_image.value_counts().to_string())

    # 5. Summary Statistics for Logging
    unique_images = df['file_name'].nunique()
    total_annotations = len(df)
    print(f"\n[Summary] Total Unique Images: {unique_images}")
    print(f"[Summary] Total Annotations: {total_annotations}")

    # Final Validation Logic
    # We expect a perfect 1:5 ratio of images to captions.
    is_balanced = (unique_images * 5 == total_annotations)
    is_clean = (missing_vals == 0 and duplicate_ids == 0 and empty_captions == 0)

    print("\n[Status] ", end="")
    if is_balanced and is_clean:
        print("PASSED: Data is completely clean and perfectly balanced (1:5 ratio).")
    else:
        print("WARNING: Data anomalies detected. Please review the checks above.")

# Execute the audit on the training data to confirm readiness for MLOps orchestration
audit_dataset(train_df, "Train")

## 3.5 Image File Integrity Audit

In a professional **MLOps** workflow, the integrity of the raw data is never assumed. When dealing with large-scale computer vision datasets—especially those sourced from compressed archives or web-scrapes—silent file corruption is a common issue. If a corrupted image is encountered during training, the PyTorch `DataLoader` will raise an exception, potentially crashing a training job that has been running for hours.

To prevent this, we perform an **Integrity Audit**. We utilize the `PIL.Image.verify()` method, which is highly efficient because it identifies whether a file is broken by reading the file headers without loading the full pixel array into memory. This allows us to scan thousands of images in seconds. Additionally, we ensure that every 'file_name' referenced in our metadata 'DataFrame' physically exists on disk to avoid `FileNotFoundError` during the training loop.

**Task: Define an `audit_image_files` function to detect missing or corrupted images and prune the metadata 'DataFrame' accordingly.**


In [ ]:
import os
from pathlib import Path
from typing import List, Tuple
from PIL import Image, UnidentifiedImageError
from tqdm.notebook import tqdm

def audit_image_files(df: pd.DataFrame, image_dir: Path) -> Tuple[List[str], List[str]]:
    """Audits image files for physical existence and file corruption.

    Args:
        df (pd.DataFrame): The DataFrame containing the 'file_name' column.
        image_dir (Path): The directory where the images are stored.

    Returns:
        Tuple[List[str], List[str]]: Lists of missing and corrupted file names.
    """
    print(f"--- Image Integrity Audit ---\n")
    print(f"Scanning directory: {image_dir}")

    unique_images = df['file_name'].unique()
    missing_files: List[str] = []
    corrupted_files: List[str] = []

    # Iterate through unique files with a progress bar
    for file_name in tqdm(unique_images, desc="Verifying Images"):
        img_path = image_dir / file_name

        # Check 1: Does the file physically exist?
        if not img_path.exists():
            missing_files.append(file_name)
            continue

        # Check 2: Can PIL open and verify the file header?
        try:
            with Image.open(img_path) as img:
                img.verify()  # Verifies that it is, in fact, an image
        except (IOError, SyntaxError, UnidentifiedImageError):
            corrupted_files.append(file_name)

    # Print Summary Report
    print("\n[Audit Summary]")
    print(f"Total Images Checked : {len(unique_images)}")
    print(f"Missing Files        : {len(missing_files)}")
    print(f"Corrupted Files      : {len(corrupted_files)}")

    if len(missing_files) == 0 and len(corrupted_files) == 0:
        print("\nPASSED: All images are physically present and uncorrupted.")
    else:
        print("\nWARNING: Broken images detected. They must be removed from the DataFrame.")

    return missing_files, corrupted_files

# Execute the image audit on the training directory
train_image_dir = CFG.TRAIN_IMG_DIR
missing_train, corrupt_train = audit_image_files(train_df, train_image_dir)

# If any bad files are found, filter them out of our DataFrame
if missing_train or corrupt_train:
    bad_files = set(missing_train + corrupt_train)
    original_len = len(train_df)

    # Drop rows where the file_name is in our list of bad files
    train_df = train_df[~train_df['file_name'].isin(bad_files)].reset_index(drop=True)

    print(f"Removed {original_len - len(train_df)} annotations attached to broken images.")

## 3.6 Creating the Custom PyTorch Dataset and DataLoader

In Deep Learning, a robust **Data Pipeline** is essential for moving data from storage to the GPU efficiently. When working with **Multimodal Tasks** like Image Captioning, we encounter a specific structural challenge: **Variable-Length Sequences**.

While images are processed through an **Image Transformation Pipeline** to reach a uniform shape (e.g., $224 \times 224$), text captions consist of different numbers of words. Since PyTorch requires all tensors in a batch to have identical dimensions, we cannot use the default collate function. Instead, we implement a **Custom Collate Function** using `pad_sequence`. This function identifies the longest caption in a specific batch and pads all other captions with a special `<PAD>` token (Index 0) to match that length.

We also implement a custom 'VizWizDataset' class inheriting from `torch.utils.data.Dataset`. This follows the standard MLOps pattern of separating data logic:
1.  `__len__`: Returns the total number of samples.
2.  `__getitem__`: Handles the "Lazy Loading" of images (loading them into RAM only when needed) and the **Numericalization** of text.

**Task:** Define the 'VizWizDataset', the 'CapsCollate' padding function, the image transformations, and instantiate the 'train_loader'.


In [ ]:
import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchvision.transforms as transforms

class VizWizDataset(Dataset):
    """
    Custom PyTorch dataset for VizWiz Image Captioning.

    This class handles the mapping between image files and their corresponding
    numericalized captions.

    Attributes:
        df (pd.DataFrame): DataFrame containing 'file_name' and 'caption'.
        img_dir (Path): Path to the folder containing the images.
        vocab (Vocabulary): Initialized Vocabulary object for numericalization.
        transform (transforms.Compose): Image transformations pipeline.
    """
    def __init__(self, df: pd.DataFrame, img_dir: Path, vocab, transform: transforms.Compose = None):
        self.df = df
        self.img_dir = img_dir
        self.vocab = vocab
        self.transform = transform

    def __len__(self) -> int:
        """Returns the total number of samples in the dataset."""
        return len(self.df)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Fetches a single image-caption pair, transforms the image,
        and converts text to a sequence of indices.
        """
        caption = self.df.iloc[index]['caption']
        img_filename = self.df.iloc[index]['file_name']
        img_path = self.img_dir / img_filename

        # Load image and convert to RGB to ensure 3 channels (removes Alpha/Grayscale)
        img = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        # Numericalize text: [START] + [indices] + [END]
        numericalized_caption = [self.vocab.stoi["<START>"]]
        numericalized_caption += self.vocab.numericalize(caption)
        numericalized_caption.append(self.vocab.stoi["<END>"])

        return img, torch.tensor(numericalized_caption)

class CapsCollate:
    """
    Custom collate function to handle variable-length sequences.

    Instead of the default stacking, this dynamically pads sequences
    within each batch to the length of the longest sequence in that batch.
    """
    def __init__(self, pad_idx: int):
        self.pad_idx = pad_idx

    def __call__(self, batch: list) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Processes a list of (image, caption) tuples into a batch.

        Args:
            batch: List of tuples from the Dataset's __getitem__.

        Returns:
            imgs: A 4D tensor of shape [Batch, Channels, Height, Width].
            targets: A 2D padded tensor of shape [Batch, Max_Seq_Len].
        """
        # Extract images and add a batch dimension for concatenation
        imgs = [item[0].unsqueeze(0) for item in batch]
        imgs = torch.cat(imgs, dim=0)

        # Extract captions
        targets = [item[1] for item in batch]

        # Pad sequences using the specified padding index (usually 0)
        targets = pad_sequence(targets, batch_first=True, padding_value=self.pad_idx)

        return imgs, targets

# --- Pipeline Configuration ---

# 1. Define standard ImageNet transformations
# We use ResNet-standard normalization values (mean and std)
transform_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 2. Instantiate the Dataset
# Assuming 'train_df', 'base_data_path', and 'vocab' are previously defined
train_image_dir = CFG.TRAIN_IMG_DIR
pad_idx = vocab.stoi["<PAD>"]

train_dataset = VizWizDataset(
    df=train_df,
    img_dir=train_image_dir,
    vocab=vocab,
    transform=transform_pipeline
)

# 3. Instantiate the DataLoader
# 'pin_memory=True' is an MLOps best practice to speed up host-to-device transfers
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,           # Shuffle to break correlation between consecutive samples
    num_workers=2,          # Use subprocesses for data loading to prevent CPU bottlenecks
    pin_memory=True,        # Optimization for faster GPU data transfer
    collate_fn=CapsCollate(pad_idx=pad_idx)
)

print(f"Total training samples : {len(train_dataset)}")
print(f"Total batches per epoch: {len(train_loader)}")

# 4. Defining the Architecture (Model 1)

## 4.1 The Encoder (Feature Extractor)

In the context of **Image Captioning**, the **Encoder** serves as the "eyes" of the system. Rather than training a model from scratch to understand shapes, colors, and textures, we utilize **Transfer Learning**. By employing a **Pre-trained Convolutional Neural Network (CNN)** like **ResNet-50**, we leverage a model that has already learned rich spatial hierarchies from the ImageNet dataset.

To adapt ResNet-50 for our needs, we perform two critical operations:
1.  **Layer Decapitation**: We remove the final fully-connected (classification) layer. We are not interested in classifying the image into one of 1000 categories; we want the high-dimensional **Feature Vector** that precedes the classification.
2.  **Weight Freezing**: We set `requires_grad = False` for the backbone parameters. This prevents the training process from destroying the finely-tuned weights of the ResNet during the early stages of training.
3.  **Linear Projection**: We add a custom 'linear' layer to project the 2048-dimensional ResNet output into a shared 'embed_size' space that matches the **Decoder's** input dimension.

**Task: Define an `EncoderCNN` class that utilizes a pre-trained ResNet-50 and projects its output to a specified embedding size**

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class EncoderCNN(nn.Module):
    """
    Extracts visual features from images using a pre-trained ResNet-50 backbone.

    The final classification layer is replaced with a linear projection to align
    with the hidden dimension of the language model.
    """
    def __init__(self, embed_size: int):
        super(EncoderCNN, self).__init__()

        # Load the pre-trained ResNet-50 model
        resnet = models.resnet50(pretrained=True)

        # Freeze the base layers to prevent backpropagation through the CNN backbone
        # This focuses training on the projection layer and the Decoder
        for param in resnet.parameters():
            param.requires_grad = False

        # Extract all modules except the final fully connected layer (fc)
        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)

        # ResNet-50's global average pooling layer outputs 2048 features.
        # We project these to the 'embed_size'.
        self.linear = nn.Linear(resnet.fc.in_features, embed_size)

        # BatchNorm is used to stabilize training and accelerate convergence
        self.batch_norm = nn.BatchNorm1d(embed_size, momentum=0.01)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """
        Performs the forward pass to transform raw pixels into feature vectors.

        Args:
            images: Input batch of images [batch_size, 3, 224, 224]

        Returns:
            Projected visual features [batch_size, embed_size]
        """
        # ResNet backbone outputs (batch_size, 2048, 1, 1)
        features = self.resnet(images)

        # Flatten the spatial dimensions to (batch_size, 2048)
        features = features.view(features.size(0), -1)

        # Project to the shared embedding space
        features = self.batch_norm(self.linear(features))

        return features

## 4.2 The Decoder (Sequence Generator)

The **Decoder** acts as the "voice" of the model, translating visual features into human-readable text. We use a **Long Short-Term Memory (LSTM)** network, a type of **Recurrent Neural Network (RNN)** designed to handle sequential data and mitigate the **Vanishing Gradient Problem**.

The generation process involves:
1.  **Word Embeddings**: Converting discrete word tokens into dense, continuous vectors using an 'nn.Embedding' layer.
2.  **Feature Integration**: We treat the image features from the **Encoder** as the very first "word" in our sequence. By concatenating the image features with the caption embeddings, we provide the LSTM with visual context.
3.  **Teacher Forcing**: During training, we pass the entire caption (minus the `<END>` token) to the LSTM. The model tries to predict the next word at each time step, and we compare these predictions against the actual next words in the caption.

**Task: Define a `DecoderRNN` class that embeds the text captions and processes them through an LSTM alongside the image features**

In [ ]:
class DecoderRNN(nn.Module):
    """
    Generates word sequences using an LSTM conditioned on visual features.

    The model takes image features and tokenized captions to predict the next
    token in the sequence via a Linear classifier over the vocabulary.
    """
    def __init__(self, embed_size: int, hidden_size: int, vocab_size: int, num_layers: int = 1):
        super(DecoderRNN, self).__init__()

        # Embedding layer to convert token indices to dense vectors
        self.embed = nn.Embedding(vocab_size, embed_size)

        # LSTM core: Processes the sequence of embeddings
        # batch_first=True ensures input/output format is (batch, seq, feature)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)

        # Final linear layer to project LSTM hidden state to vocabulary size
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, features: torch.Tensor, captions: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for sequence generation/training.

        Args:
            features: Image feature vectors [batch_size, embed_size]
            captions: Tokenized target captions [batch_size, seq_len]

        Returns:
            Logits for each word in the sequence [batch_size, seq_len, vocab_size]
        """
        # Remove the <END> token to maintain alignment during training
        captions = captions[:, :-1]

        # Transform tokens to embeddings: [batch_size, seq_len-1, embed_size]
        embeddings = self.embed(captions)

        # Concatenate image features as the first 'token' in the sequence
        # features.unsqueeze(1) changes shape to [batch_size, 1, embed_size]
        embeddings = torch.cat((features.unsqueeze(1), embeddings), dim=1)

        # hiddens shape: [batch_size, seq_len, hidden_size]
        hiddens, _ = self.lstm(embeddings)

        # Map LSTM outputs to vocabulary logits
        outputs = self.linear(hiddens)

        return outputs

## 4.3 Model Assembly and Device Mapping

To ensure a clean and scalable **MLOps** workflow, we wrap the **Encoder** and **Decoder** into a unified `ImageCaptioningModel`. This high-level wrapper simplifies the training loop by abstracting the flow of data between the CNN and RNN.

During this phase, we also handle **Hardware Acceleration**. We check for the availability of a **CUDA-enabled GPU** to significantly speed up the matrix multiplications required for both ResNet and LSTM operations. Finally, we perform a **Parameter Audit** to distinguish between total parameters and trainable parameters—crucial for verifying that our CNN backbone is correctly frozen.

**Task: Define the wrapper class, set the structural hyperparameters, and instantiate the model to the target compute device.**

In [ ]:
class ImageCaptioningModel(nn.Module):
    """
    Unified interface for the Encoder-Decoder architecture.
    Handles the end-to-end flow from image input to caption logits.
    """
    def __init__(self, encoder: nn.Module, decoder: nn.Module):
        super(ImageCaptioningModel, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, images: torch.Tensor, captions: torch.Tensor) -> torch.Tensor:
        """
        Coordinates the visual extraction and sequential generation.
        """
        # Step 1: Extract visual context
        features = self.encoder(images)

        # Step 2: Generate sequence predictions based on context and captions
        outputs = self.decoder(features, captions)

        return outputs

# --- Architectural Hyperparameters ---
EMBED_SIZE = 256    # Dimension of the shared visual/textual space
HIDDEN_SIZE = 512   # Number of units in the LSTM hidden state
NUM_LAYERS = 1      # Number of LSTM layers stacked
VOCAB_SIZE = len(vocab) # Assumes 'vocab' object exists from Section 3

# --- Device Configuration ---
# Move the model to GPU if available for high-performance training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target compute device: {device}")

# --- Model Instantiation ---
encoder_net = EncoderCNN(embed_size=EMBED_SIZE)
decoder_net = DecoderRNN(
    embed_size=EMBED_SIZE,
    hidden_size=HIDDEN_SIZE,
    vocab_size=VOCAB_SIZE,
    num_layers=NUM_LAYERS
)

# Combine and transfer to device
cap_model = ImageCaptioningModel(encoder_net, decoder_net).to(device)

# --- Parameter Audit ---
total_params = sum(p.numel() for p in cap_model.parameters())
trainable_params = sum(p.numel() for p in cap_model.parameters() if p.requires_grad)

print(f"\nTotal Parameters: {total_params:,}")
print(f"\nTrainable Parameters: {trainable_params:,}")
print(f"\n{cap_model}")

## 4.4 Defining the Loss and Optimizer

In sequence-to-sequence tasks like image captioning, we treat the problem as a multi-class classification task at each time step. We use **Cross-Entropy Loss** to measure the discrepancy between the predicted probability distribution over the vocabulary and the actual ground-truth word.

A critical implementation detail in NLP is handling **Padding Tokens** ('<PAD>'). Since sequences in a batch are padded to the same length, we do not want the model to be penalized or rewarded for predicting these placeholder tokens. By setting the `ignore_index` parameter in our loss function, we ensure that the gradients are not calculated for padding positions, allowing the model to focus its learning capacity on meaningful language patterns.

For optimization, we utilize the **Adam (Adaptive Moment Estimation)** optimizer. Because we are using a transfer learning approach with a pre-trained **ResNet** backbone, we use a `filter` function to pass only the parameters that have `requires_grad = True` to the optimizer. This prevents unnecessary computation and potential corruption of the frozen feature extractor layers.



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from typing import Iterable

# Retrieve the padding index from our vocabulary to ensure the loss function ignores it
# Variable 'pad_idx' is used to mask the loss during calculation
pad_idx: int = vocab.stoi["<PAD>"]

# Define the loss function.
# We use CrossEntropyLoss which combines LogSoftmax and NLLLoss in one single class.
criterion: nn.Module = nn.CrossEntropyLoss(ignore_index=pad_idx)

# Define the optimizer.
# We filter 'cap_model.parameters()' to only include those that require gradients.
# This is essential when the Encoder (CNN) is frozen.
trainable_params: Iterable = filter(lambda p: p.requires_grad, cap_model.parameters())

optimizer: optim.Optimizer = optim.Adam(
    trainable_params,
    lr=CFG.LEARNING_RATE
)

print(f"Loss Function: {criterion}")
print(f"Optimizer: Adam (LR: {CFG.LEARNING_RATE})")

## 4.5 The Training Loop Implementation

The training loop is the core of the MLOps pipeline where the model learns to map visual features to textual sequences. We employ a technique known as **Teacher Forcing**. During training, instead of using the model's own (potentially incorrect) prediction from the previous time step as the next input, we feed it the actual ground-truth caption.

To implement this, we perform a specific alignment:
1.  **Input Sequence**: Captions excluding the final '<END>' token.
2.  **Target Sequence**: Captions excluding the initial '<START>' token.

This forces the model to predict the "next" word in the sequence given the image and the preceding correct words. During the **Forward Pass**, the model produces a tensor of shape `(batch_size, sequence_length, vocab_size)`. To calculate the loss, we must "flatten" these tensors into a 2D format for the predictions and a 1D format for the targets, as expected by PyTorch's `CrossEntropyLoss`.

In [ ]:
import time
from typing import List

def train_model(
    model: nn.Module,
    data_loader: torch.utils.data.DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    num_epochs: int,
    device: torch.device
) -> List[float]:
    """
    Trains the Image Captioning model and returns the loss history.

    Args:
        model: The Encoder-Decoder architecture to train.
        data_loader: The DataLoader providing batches of (images, captions).
        criterion: The loss function (CrossEntropyLoss).
        optimizer: The optimization algorithm (Adam).
        num_epochs: Number of iterations over the full dataset.
        device: The hardware to run training on (cuda or cpu).
    """
    print(f"--- Starting Training on {device} ---")
    model.train()

    epoch_losses: List[float] = []

    for epoch in range(num_epochs):
        start_time: float = time.time()
        running_loss: float = 0.0

        for batch_idx, (images, captions) in enumerate(data_loader):
            # Move tensors to the configured 'device'
            images = images.to(device)
            captions = captions.to(device)

            # Reset gradients to zero for every batch to prevent accumulation
            optimizer.zero_grad()

            # Forward pass:
            # The model processes images and the 'input' portion of captions
            outputs = model(images, captions)

            # Target shifting: Predict the NEXT word.
            # We remove the <START> token (index 0) from the targets.
            # 'targets' shape: (batch_size, seq_len - 1)
            targets = captions[:, 1:]

            # Reshape outputs to (N*seq_len, vocab_size) and targets to (N*seq_len)
            # This flattening is required for the CrossEntropyLoss function
            loss = criterion(
                outputs.reshape(-1, outputs.shape[2]),
                targets.reshape(-1)
            )

            # Backward pass: Compute gradients of the loss w.r.t. model parameters
            loss.backward()

            # Gradient clipping is often recommended for RNNs to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            # Optimizer step: Update weights based on computed gradients
            optimizer.step()

            running_loss += loss.item()

            # Periodic logging for monitoring training convergence
            if (batch_idx + 1) % 500 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx+1}/{len(data_loader)}], Loss: {loss.item():.4f}")

        # Calculate and store average loss for the epoch
        avg_epoch_loss = running_loss / len(data_loader)
        epoch_losses.append(avg_epoch_loss)
        elapsed_time = time.time() - start_time

        print(f"==> Epoch [{epoch+1}/{num_epochs}] completed in {elapsed_time:.2f}s | Average Loss: {avg_epoch_loss:.4f}\n")

    return epoch_losses

# Execute the training process
# Note: 'cap_model', 'train_loader', and 'device' must be defined in previous cells
train_losses = train_model(
    model=cap_model,
    data_loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=CFG.EPOCHS,
    device=device
)

# 5. Evaluation (Model 1)

In this phase, we move beyond the training loop to assess how well our model has learned to map visual features to linguistic sequences. Evaluation in image captioning is unique because there isn't one "correct" answer; a single image can be described in many valid ways. We will employ both **quantitative metrics** and **qualitative visual inspection** to judge the performance of our 'model'.

## 5.1 The Inference Engine

During the training phase, we utilized a technique called **Teacher Forcing**, where the ground-truth caption is fed into the decoder to predict the next token. However, during **Inference** (production), we do not have the ground truth.

The model must operate **autoregressively**:
1. It receives an image feature vector from the **Encoder**.
2. It is primed with a special `'<START>'` token.
3. It predicts the probability distribution of the next word.
4. We use **Greedy Search** to select the word with the highest probability.
5. This predicted word is fed back into the model as the input for the next time step.
6. This cycle repeats until the model generates an `'<END>'` token or reaches the 'max_len'.

**Task: Define a function 'generate_caption' that performs inference on a single image.**



In [ ]:
import torch
import torch.nn as nn
from typing import List

def generate_caption(
    model: nn.Module,
    image: torch.Tensor,
    vocab: 'Vocabulary',
    max_len: int = 20
) -> List[str]:
    """
    Generates a caption for an image using greedy search.

    Args:
        model (nn.Module): The trained Image Captioning model.
        image (torch.Tensor): The preprocessed image tensor of shape (C, H, W).
        vocab (Vocabulary): The vocabulary mapping object.
        max_len (int): Maximum length of the generated caption to prevent infinite loops.

    Returns:
        List[str]: A list of predicted words (strings) representing the caption.
    """
    # Set model to evaluation mode (disables dropout, batchnorm, etc.)
    model.eval()
    result_caption = []

    # Disable gradient calculation for memory efficiency and speed
    with torch.no_grad():
        # Add batch dimension: (1, C, H, W) and move to the target device
        image = image.unsqueeze(0).to(device)

        # Pass image through the Encoder to get feature vectors
        features = model.encoder(image)

        # Initialize the hidden and cell states for the LSTM
        states = None

        # Start the sequence with the <START> token index
        start_idx = vocab.stoi["<START>"]
        current_word_idx = torch.tensor([start_idx]).to(device)

        for _ in range(max_len):
            # 1. Pass the current word through the Embedding layer
            embeddings = model.decoder.embed(current_word_idx).unsqueeze(0)

            # 2. Forward pass through LSTM with previous hidden states
            hiddens, states = model.decoder.lstm(embeddings, states)

            # 3. Project LSTM output to vocabulary size
            outputs = model.decoder.linear(hiddens.squeeze(0))

            # 4. Greedy Search: Pick the index with the highest logit/probability
            predicted_idx = outputs.argmax(1)

            # 5. Map index back to string via the vocabulary
            predicted_word = vocab.itos[predicted_idx.item()]

            # Stop condition: If model predicts the end of the sentence
            if predicted_word == "<END>":
                break

            result_caption.append(predicted_word)

            # Update the current_word_idx for the next iteration
            current_word_idx = predicted_idx

    return result_caption

## 5.2 Calculating BLEU Scores

To evaluate the model's accuracy at scale, we use the **BLEU (Bilingual Evaluation Understudy)** score. This is a standard NLP metric that calculates the **n-gram overlap** between the model's generated 'hypothesis' and one or more human-written 'references'.

*   **BLEU-1 (Unigrams):** Measures how many individual words match. It focuses on vocabulary accuracy.
*   **BLEU-4 (4-grams):** Measures how many 4-word sequences match. It serves as a proxy for grammatical fluency and coherence.

In this step, we iterate through the validation set, generate a hypothesis for each image, and compare it against the five reference captions provided by humans.

**Task: Iterate through a validation sample, generate captions, and calculate the corpus-level BLEU scores using 'nltk'.**

In [ ]:
import nltk
import pandas as pd
import random
from pathlib import Path
from PIL import Image
from nltk.translate.bleu_score import corpus_bleu

# Download NLTK data required for tokenization
nltk.download('punkt', quiet=True)

def evaluate_bleu(
    model: nn.Module,
    val_df: pd.DataFrame,
    image_dir: Path,
    vocab: 'Vocabulary',
    num_samples: int = 500
):
    """
    Calculates BLEU 1-4 scores against the validation dataset by comparing
    generated captions to human references.
    """
    print(f"--- Evaluating BLEU Scores on {num_samples} Validation Samples ---")

    references_corpus = []
    hypotheses_corpus = []

    # Group validation data by image to gather all 5 human references per image
    grouped = val_df.groupby('file_name')
    unique_files = list(grouped.groups.keys())

    # Select a random subset to speed up evaluation during development
    sample_files = random.sample(unique_files, min(num_samples, len(unique_files)))

    for file_name in sample_files:
        img_path = image_dir / file_name
        if not img_path.exists():
            continue

        # 1. Prepare references: Tokenize all 5 human captions for this image
        group = grouped.get_group(file_name)
        references = [vocab.tokenize(row['caption']) for _, row in group.iterrows()]
        references_corpus.append(references)

        # 2. Prepare hypothesis: Load image and generate caption via model
        img = Image.open(img_path).convert("RGB")
        img_tensor = transform_pipeline(img) # Assuming transform_pipeline is pre-defined

        hypothesis = generate_caption(model, img_tensor, vocab)
        hypotheses_corpus.append(hypothesis)

    # 3. Calculate BLEU scores using specific weights for n-grams
    # weights=(1.0, 0, 0, 0) -> BLEU-1
    # weights=(0.25, 0.25, 0.25, 0.25) -> BLEU-4
    bleu_1 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(1.0, 0, 0, 0))
    bleu_2 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.5, 0.5, 0, 0))
    bleu_3 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.33, 0.33, 0.33, 0))
    bleu_4 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.25, 0.25, 0.25, 0.25))

    print("\n[Evaluation Results]")
    print(f"BLEU-1 Score: {bleu_1:.4f}")
    print(f"BLEU-2 Score: {bleu_2:.4f}")
    print(f"BLEU-3 Score: {bleu_3:.4f}")
    print(f"BLEU-4 Score: {bleu_4:.4f}")

    return bleu_1, bleu_2, bleu_3, bleu_4

# Execution Note: Ensure val_df and val_image_dir are defined in your workspace
bleu_results = evaluate_bleu(cap_model, val_df, val_image_dir, vocab)

## 5.3 Visual Inspection

Metrics like **BLEU** are strictly mathematical and often fail to capture the semantic nuance of language. For example, a model might describe a "dog" as a "canine"; the BLEU score would penalize this as a mismatch, even though it is factually correct.

**Visual Inspection** allows the researcher to see if the model's errors are "logical" (e.g., misidentifying a specific breed of dog) or "hallucinatory" (e.g., seeing a car where there is none).

**Task: Plot random validation images alongside our model's generated caption and the actual human references.**

In [ ]:
import matplotlib.pyplot as plt

def inspect_model_visually(
    model: nn.Module,
    val_df: pd.DataFrame,
    image_dir: Path,
    vocab: 'Vocabulary',
    num_samples: int = 3
):
    """
    Displays images from the validation set along with the predicted
    caption and all ground truth references for qualitative analysis.
    """
    grouped = val_df.groupby('file_name')
    unique_files = list(grouped.groups.keys())
    sample_files = random.sample(unique_files, min(num_samples, len(unique_files)))

    for file_name in sample_files:
        img_path = image_dir / file_name
        if not img_path.exists():
            continue

        # Generate prediction using the previously defined inference engine
        img = Image.open(img_path).convert("RGB")
        img_tensor = transform_pipeline(img)
        predicted_words = generate_caption(model, img_tensor, vocab)
        predicted_sentence = " ".join(predicted_words).capitalize() + "."

        # Setup Visualization
        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.show()

        # Display text output with formatting for readability
        print(f"--- Visual Inspection: {file_name} ---")
        print(f"\033[1mPredicted: \033[92m{predicted_sentence}\033[0m") # Bold and Green text
        print("\nGround Truth References:")

        # Retrieve and print all human references for this specific image
        group = grouped.get_group(file_name)
        for i, (_, row) in enumerate(group.iterrows(), 1):
            print(f"  {i}. {row['caption']}")
        print("="*60 + "\n")

# Execute visual inspection to check model quality qualitatively
inspect_model_visually(cap_model, val_df, val_image_dir, vocab)